In [3]:
from collections.abc import Sequence
from typing import cast

import matplotlib
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as F
import torchvision
from PIL import Image
from torchvision.utils import make_grid

import sys,os
from pathlib import Path
print(str(Path(os.getcwd()).resolve().parent.parent))
sys.path.append(str(Path(os.getcwd()).resolve().parent.parent))

from shimmer.modules.global_workspace import (
    GlobalWorkspaceFusion,
)

from shimmer_metaworld import DEBUG_MODE, PROJECT_DIR,LOGGER
from shimmer_metaworld.config import load_config
from shimmer_metaworld.logging import get_pil_image, batch_to_device
from metaworld_dataset import (
    MetaworldDataModule,
    get_default_domains,
)
from shimmer_metaworld.modules.domains import load_pretrained_domains

matplotlib.use("Agg")

no_act = {
    "gw_ckpt" : "9f74du8u",
    "ckpt_epoch" : "26"
}
act = {
    "gw_ckpt" : "kp56fk1e",
    "ckpt_epoch" : "194"
}
gw_ckpt = act["gw_ckpt"]
ckpt_epoch = act["ckpt_epoch"]

/home/yelhelw/metaworld_GW/shimmer-metaworld


In [5]:

print(gw_ckpt)
def image_grid_from_v_tensor(
    samples: Sequence[torch.Tensor],
    _: int,
    ncols: int,
) -> Image:
    image = make_grid(samples[0], nrow=ncols, pad_value=1).detach()
    return F.to_pil_image(image)


debug_mode = DEBUG_MODE
extra_config_files = ["train_gw.yaml"]
argv = []

LOGGER.debug(f"Debug mode: {debug_mode}")

config = load_config(
    PROJECT_DIR / "shimmer_metaworld"/ "config_template",
    load_files=extra_config_files,
    debug_mode=debug_mode,
    log_config=False,
    argv=argv,
)

#seed_everything(config.seed, workers=True)

domain_classes = get_default_domains(
    {domain.domain_type.kind.value for domain in config.domains}
)
print(config.domains)
domain_modules, gw_encoders, gw_decoders = load_pretrained_domains(
    config.domains,
    config.global_workspace.latent_dim,
    config.global_workspace.encoders.hidden_dim,
    config.global_workspace.encoders.n_layers,
    config.global_workspace.decoders.hidden_dim,
    config.global_workspace.decoders.n_layers,
    is_linear=config.global_workspace.linear_domains,
    bias=config.global_workspace.linear_domains_use_bias,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = f'/mnt/datashare/yelhelw/shimmer-meta-{gw_ckpt}/epoch={ckpt_epoch}.ckpt'

domain_module = GlobalWorkspaceFusion.load_from_checkpoint(ckpt_path, domain_mods=domain_modules,
    gw_encoders=gw_encoders,
    gw_decoders=gw_decoders)
domain_module.eval().freeze()

domain_module.to(device)

kp56fk1e
[LoadedDomainConfig(checkpoint_path=PosixPath('pretrained/domain_v.ckpt'), domain_type=<DomainModuleVariant.v_latents: (<DomainType.v_latents: DomainDesc(base='v', kind='v_latents')>, 'default')>, args={}), LoadedDomainConfig(checkpoint_path=PosixPath('pretrained/domain_attr.ckpt'), domain_type=<DomainModuleVariant.attr: (<DomainType.attr: DomainDesc(base='attr', kind='attr')>, 'default')>, args={}), LoadedDomainConfig(checkpoint_path=PosixPath('/mnt/datashare/yelhelw/checkpoints/shimmer-meta-7zc4r0xq/epoch=63.ckpt'), domain_type=<DomainModuleVariant.act: (<DomainType.act: DomainDesc(base='act', kind='act')>, 'default')>, args={})]


GlobalWorkspaceFusion(
  (gw_mod): GWModule(
    (domain_mods): ModuleDict(
      (v_latents): VisualLatentDomainModule(
        (visual_module): VisualDomainModule(
          (vae): VAE(
            (encoder): RAEEncoder(
              (layers): Sequential(
                (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4), bias=False)
                (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (2): ReLU()
                (3): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
                (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (5): ReLU()
                (6): Conv2d(128, 257, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
                (7): BatchNorm2d(257, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (8): ReLU()
                (9): Conv2d(257, 514, kernel_size=(4, 4), stride=(2, 

In [3]:

for name, module in domain_module.gw_mod.gw_decoders.items():
    print(name,"->",module)


v_latents -> GWDecoder(
  (0): Linear(in_features=12, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=32, bias=True)
  (7): ReLU()
  (8): Linear(in_features=32, out_features=16, bias=True)
)
attr -> GWDecoder(
  (0): Linear(in_features=12, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=32, bias=True)
  (7): ReLU()
  (8): Linear(in_features=32, out_features=17, bias=True)
)


In [6]:
#Load data
domain_classes = get_default_domains(["v_latents","act", "attr"])

print(config.domain_proportions)
data_module = MetaworldDataModule(
        '/mnt/datashare/yelhelw/',
        domain_classes,
        config.domain_proportions,
        batch_size=config.training.batch_size,
        num_workers=config.training.num_workers,
        seed=config.seed,
        ood_seed=config.ood_seed,
        domain_args=config.domain_data_args,
    )


{frozenset({'v', 'attr'}): 1.0, frozenset({'v', 'act', 'attr'}): 0.4}


__Action to vision grid__ 

In [5]:
train_samples = data_module.get_samples("train",32,offset=1)
train_samples=batch_to_device(train_samples,device)
import numpy as np
img_latents = np.load("/mnt/datashare/yelhelw/saved_latents/train/domain_v.npy")
act = domain_module.encode_domain(train_samples[frozenset(["act","attr"])]["act"],"act")
print(len(act))
#latent_img_uni = domain_module.encode_domain(img_latents, "v_latents")
gw_latent_v = domain_module.gw_mod.encode({"v_latents": torch.from_numpy(img_latents).to(device)})
gw_latent_act = domain_module.gw_mod.encode({"act":act})

KeyError: frozenset({'act', 'attr'})

In [4]:
import itertools
import torch
import numpy as np
# Generate all 27 combinations of (-1, 0, 1)^3
#combos = list(itertools.product([-1,0,1,], repeat=4))
combos = []
index = 0
for x in np.arange(-1,1,.25):
    for z in np.arange(-1,1,.25):
        for gripper in np.arange(0,1,.25):
            combos.append([x,1,z,gripper])
            #print(combos[index])
            index +=1
# Convert to torch tensor and reshape into 3×3×3×3
act = torch.tensor(combos).to(device)
#act[:,2]=0.6
print(act)

gw_latent_act = domain_module.gw_mod.encode({"act":act})
gw_latent_v_fused = domain_module.gw_mod.fuse(gw_latent_act, {"act": torch.ones(gw_latent_act['act'].size(0)).to(device)})
decoded_latent_uni = domain_module.gw_mod.decode(gw_latent_v_fused)



tensor([[-1.0000,  1.0000, -1.0000,  0.0000],
        [-1.0000,  1.0000, -1.0000,  0.2500],
        [-1.0000,  1.0000, -1.0000,  0.5000],
        ...,
        [ 0.7500,  1.0000,  0.7500,  0.2500],
        [ 0.7500,  1.0000,  0.7500,  0.5000],
        [ 0.7500,  1.0000,  0.7500,  0.7500]], device='cuda:0',
       dtype=torch.float64)


KeyError: 'act'

In [6]:
from torchvision.utils import make_grid,save_image
decoded_images = domain_modules['v_latents'].decode_images(decoded_latent_uni['v_latents'])

from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms as T
from torchvision.utils import make_grid

to_pil = T.ToPILImage()
to_tensor = T.ToTensor()

labeled_images = []

for img_tensor, label in zip(decoded_images, act):
    img = to_pil(img_tensor.cpu())
    draw = ImageDraw.Draw(img)

    # Convert tensor label (e.g., tensor([-1,0,1,1])) → string
    label_text = str(label.tolist())

    # Draw white text in top-left corner
    draw.text((5, 5), label_text, fill="white")

    # Back to tensor
    labeled_images.append(to_tensor(img))

# Stack and grid
labeled_images = torch.stack(labeled_images)

images = make_grid(labeled_images, pad_value=1)
save_image(images, "grid_output.png")

NameError: name 'decoded_latent_uni' is not defined

In [ ]:
decoded_attributes = domain_module.decode_domain(decoded_latent_uni["attr"],"attr")
print(decoded_attributes)

__Object-dependent visualization__

In [7]:
object = "obj"
object_images_path = Path(f"/home/yelhelw/metaworld_GW/Myworld/tasks/objects/{object}")

images = []
for x in range(1000):
    path = object_images_path/f"{x}.png"
    #path = self.image_path / f"{index}.png"
    with Image.open(path) as image:
        image = image.convert("RGB")
    tensor_img = torchvision.transforms.functional.pil_to_tensor(image).to(device).to(dtype=torch.float32) / 255
    images.append(tensor_img)
images = torch.stack(images)

latent_v_wall = domain_modules['v_latents'].visual_module.vae.encoder(images[:])[0]
print(latent_v_wall.shape)

torch.Size([1000, 16])


In [8]:
object_images_path = Path(f"/home/yelhelw/metaworld_GW/Myworld/tasks/objects/no_{object}")
import torchvision
images = []
for x in range(1000):
    path = object_images_path/f"{x}.png"
    #path = self.image_path / f"{index}.png"
    with Image.open(path) as image:
        image = image.convert("RGB")
    tensor_img = torchvision.transforms.functional.pil_to_tensor(image).to(device).to(dtype=torch.float32) / 255
    images.append(tensor_img)
images = torch.stack(images)

latent_v_no_wall = domain_modules['v_latents'].visual_module.vae.encoder(images[:])[0]
print(latent_v_no_wall.shape)

torch.Size([1000, 16])


In [9]:
gw_latent_v_wall = domain_module.gw_mod.encode({"v_latents":latent_v_wall})
gw_latent_v_no_wall = domain_module.gw_mod.encode({"v_latents":latent_v_no_wall})

In [10]:
gw_latent_v_wall_fused = domain_module.gw_mod.fuse(gw_latent_v_wall, {"v_latents": torch.ones(gw_latent_v_wall['v_latents'].size(0)).to(device)})
gw_latent_v_no_wall_fused = domain_module.gw_mod.fuse(gw_latent_v_no_wall, {"v_latents": torch.ones(gw_latent_v_no_wall['v_latents'].size(0)).to(device)})

__Latent UMAP structures__

In [11]:
from collections.abc import Mapping
from shimmer.modules.selection import FixedSharedSelection
from matplotlib.colors import ListedColormap
from tqdm import tqdm


def to_device(data: torch.Tensor | Mapping[str, torch.Tensor] | list, device: str):
    """Put the data Tensor or list on the device (GPU or CPU)"""
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, list):
        return [value.to(device) for value in data]
    elif isinstance(data, Mapping):
        return {name: to_device(value, device) for name, value in data.items()}
    else:
        raise TypeError(f"Unsupported type: {type(data)}")


In [12]:
import numpy as np

modalities = ["v","act","attr"]
keys = frozenset({"v_latents","act","attr"})
fuse = True

np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fusion_mech = FixedSharedSelection()

data_module.prepare_data()
data_module.setup()

data_module.val_dataset = {keys :data_module.val_dataset[keys]}

dataloaders = {
    #"train": data_module.train_dataloader(shuffle=False, drop_last=False),
    "val" : data_module.val_dataloader(),
}

for split, dataloader in dataloaders.items():
    latents: list[np.ndarray] = []
    latents_dict : dict = {m: [] for m in modalities}
    print(f"Saving {split}.")
    for batch, _, _ in tqdm(iter(dataloader), total=len(dataloader)):
        data = to_device(batch[keys], device)
   
        latent = {}
        for modality in modalities:
            if modality == 'v':
                latent['v_latents'] = domain_modules['v_latents'].encode(data['v_latents'])
            else:
                latent[modality] = domain_modules[modality].encode(data[modality])
        else:
            latent = domain_module.gw_mod.encode(latent)
            if fuse:
                selection_scores = fusion_mech(latent, latent)
                latent_fuse = domain_module.gw_mod.fuse(latent, selection_scores)
                latents.append(latent_fuse.detach().cpu().numpy())

                latent['v'] = latent.pop('v_latents')
                latents_dict = {m: latents_dict[m] + [torch.tanh(latent[m]).detach().cpu().numpy()] for m in latents_dict.keys() if m in latent}
            else:
                latent['v'] = latent.pop('v_latents')
                latents_dict = {m: latents_dict[m] + [latent[m].detach().cpu().numpy()] for m in latents_dict.keys() if m in latent}

    if len(latents) > 0:
        latent_vectors = np.concatenate(latents, axis=0)
        shuffle_latent_vectors = latent_vectors.copy()
        np.random.shuffle(shuffle_latent_vectors)
        latents_dict = {m: np.concatenate(v, axis=0) for m, v in latents_dict.items()}
        latent_vectors = np.concatenate([latents_dict[m] for m in latents_dict.keys()], axis=0)
    else:
        latents_dict = {m: np.concatenate(v, axis=0) for m, v in latents_dict.items()}
        latent_vectors = np.concatenate([latents_dict[m] for m in latents_dict.keys()], axis=0)
        shuffle_latent_vectors = latent_vectors.copy()
        np.random.shuffle(shuffle_latent_vectors)

    print(len(latents_dict['v']))

Saving val.


/home/yelhelw/metaworld_GW/shimmer-metaworld/metaworld_dataset/domain_alignment.py:94: UserWarning: Domains have different lengths. Selecting min (160000).
  dataset = MetaworldDataset(
/home/yelhelw/metaworld_GW/shimmer-metaworld/metaworld_dataset/domain_alignment.py:94: UserWarning: Domains have different lengths. Selecting min (40040).
  dataset = MetaworldDataset(
100%|██████████| 313/313 [00:04<00:00, 65.88it/s]

40040


In [13]:
np.save("shuffle_latent_vectors_act.npy",shuffle_latent_vectors)
np.save("gw_latent_v_ball_fused_act.npy",gw_latent_v_wall_fused.detach().cpu().numpy())
np.save("gw_latent_v_no_ball_fused_act.npy",gw_latent_v_no_wall_fused.detach().cpu().numpy())

<p align="center">
    UMAP parameters sweep
</p>

In [ ]:
from umap.umap_ import nearest_neighbors
import pickle



knn = nearest_neighbors(shuffle_latent_vectors,
                              n_neighbors=100,
                              metric="cosine",
                              metric_kwds=None,
                              angular=True,
                              random_state=None,
                             )

with open("knn_ball_no_action.pkl","wb") as f:
    pickle.dump(knn,f)

In [12]:
import pickle
with open("knn_no_action.pkl", "rb") as f:
    knn = pickle.load(f)

In [ ]:
import umap

n_neighbors = [5, 25,50, 100]
min_dists = [0, 0.2, 0.5, 0.9]

embeddings = np.zeros((2,4, 4, 1000, 2))
for i, k in enumerate(n_neighbors):
    for j, dist in enumerate(min_dists):
        print(k,dist)
        reducer = umap.UMAP(n_neighbors=k,
                                                      min_dist=dist,
                                                      precomputed_knn=knn,random_state=42
                                                      ).fit(shuffle_latent_vectors)
        print("reducer done")
        embeddings[1,i, j] = reducer.transform(gw_latent_v_wall_fused.detach().cpu().numpy())
        print("first embedding")
        embeddings[0,i, j] = reducer.transform(gw_latent_v_no_wall_fused.detach().cpu().numpy())

NameError: name 'np' is not defined

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(4, 4, figsize=(20, 20))

for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        ax.scatter(embeddings[0,i, j, :, 0],
                   embeddings[0,i, j, :, 1],
                   c="green",
                   alpha=0.8,
                   s=1,
                   )
        ax.scatter(embeddings[1,i, j, :, 0],
                   embeddings[1,i, j, :, 1],
                   c="red",
                   alpha=0.8,
                   s=1,
                   )
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title("min_dist = {}".format(min_dists[j]), size=15)
        if j == 0:
            ax.set_ylabel("n_neighbors = {}".format(n_neighbors[i]), size=15)
fig.suptitle("UMAP embedding of MNIST digits with grid of parameters", y=0.92, size=20)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.savefig("graphs/no_act/UMAP_sweep.png")
plt.close()

In [48]:
import umap
#reducer = umap.UMAP(n_neighbors=2, min_dist=0, spread=3, n_components=2,verbose = True)
#reducer = umap.UMAP(n_neighbors=100,min_dist=0.2,precomputed_knn=knn).fit(shuffle_latent_vectors)
reducer = umap.UMAP(n_neighbors=100,min_dist=0.2,metric="cosine",metric_kwds=None).fit(shuffle_latent_vectors)

In [32]:
import pickle

with open("umap_model_cosine_100_0.2.pkl","wb") as f:
    pickle.dump(reducer,f)

'''
with open("umap_model_norm.pkl", "rb") as f:
    reducer = pickle.load(f)
'''


'\nwith open("umap_model_norm.pkl", "rb") as f:\n    reducer = pickle.load(f)\n'

<p align="center">
    Object centered Umap figures
</p>

In [49]:
embedding_wall = reducer.transform(gw_latent_v_wall_fused.detach().cpu().numpy())
embedding_no_wall = reducer.transform(gw_latent_v_no_wall_fused.detach().cpu().numpy())

In [58]:
fig = plt.figure()
plt.scatter(embedding_wall[:,0],embedding_wall[:,1],color="green",s=1,label="wall")
plt.scatter(embedding_no_wall[:,0],embedding_no_wall[:,1],color="red",s=1,label="no_wall")
plt.legend()
plt.savefig('graphs/no_act/Umap_wall.png')


In [51]:
embedding_wall_vae = reducer.transform(latent_v_wall.detach().cpu().numpy())
embedding_no_wall_vae = reducer.transform(latent_v_no_wall.detach().cpu().numpy())

In [52]:
fig = plt.figure()
plt.scatter(embedding_wall_vae[:,0],embedding_wall_vae[:,1],color="green",label="wall")
plt.scatter(embedding_no_wall_vae[:,0],embedding_no_wall_vae[:,1],color="red",label="no_wall")
plt.legend()
plt.savefig('graphs/no_act/Umap_vae_wall_cosine.png')

<p align="center">
    General Embeddings
</p>

In [25]:

labels = np.load('/mnt/datashare/yelhelw/actions_val.npy', mmap_mode="r")
all_labels = np.repeat(labels, 3, axis=0)

n_samples_per_modality = len(labels)
modality_labels = []
for i, modality in enumerate(modalities):
    modality_labels.extend([i] * n_samples_per_modality)
modality_labels = np.array(modality_labels)

modality_names = {0: 'Vision (v)', 1: 'Attributes (attr)', 2: 'Actions (act)'}
x_disp = np.clip(labels[:, 0],-1,1)
y_disp = np.clip(labels[:, 1],-1,1)
z_disp = np.clip(labels[:, 2],-1,1)
gripper = np.clip(labels[:, 3],-1,1)

embedding_all = reducer.transform(latent_vectors)
#np.save("umpa_embedding_all_val.npy",embedding_all)


In [23]:
#Embeddings per modality
embedding_mod = {}
print(latents_dict.keys())
modalities = ["v","attr", "act"]
for m in modalities:
    embedding = reducer.transform(latents_dict[m])
    embedding_mod[m] = embedding
#np.save("umpa_embedding_mod_val.npy",embedding_mod)

dict_keys(['v', 'attr', 'act'])


In [ ]:
embedding_mod = np.load("umpa_embedding_mod_val.npy",allow_pickle=True).item()
embedding_all = np.load("umpa_embedding_all_final.npy",allow_pickle=True)

In [28]:

modalities = ["v","attr", "act"]
modality_cmap = ListedColormap(['red', 'green', 'orange'])
all_embeddings = [embedding_all] + list(embedding_mod.values())
x_min = min([emb[:, 0].min() for emb in all_embeddings])
x_max = max([emb[:, 0].max() for emb in all_embeddings])
y_min = min([emb[:, 1].min() for emb in all_embeddings])
y_max = max([emb[:, 1].max() for emb in all_embeddings])

x_range = x_max - x_min
y_range = y_max - y_min
padding = 0.05
x_limits = [x_min - padding * x_range, x_max + padding * x_range]
y_limits = [y_min - padding * y_range, y_max + padding * y_range]

actions = ['right-left', 'front-back', 'up-down', 'gripper']
n_actions = len(actions)
n_modalities = len(modalities)

n_cols = n_modalities
n_rows = n_actions + 1

fig, axes = plt.subplots(n_rows, n_cols + 1, figsize=(6 * n_cols + 2, 4 * n_rows))
plt.subplots_adjust(hspace=0.3, wspace=0.3, right=0.92)

###################################
# Top row: Modality plot centered #
###################################
axes[0, 0].remove()
axes[0, 2].remove()

modality_ax = axes[0, 1]
scatter_mod = modality_ax.scatter(
    embedding_all[:480000, 0],
    embedding_all[:480000, 1],
    c=modality_labels,
    cmap=modality_cmap,
    s=5,
    alpha=0.2
)
modality_ax.set_title('UMAP - All Modalities by Type', fontsize=16, fontweight='bold')
modality_ax.set_xlim(x_limits)
modality_ax.set_ylim(y_limits)

modality_legend_elements = []
for i, color in enumerate(modality_cmap.colors):
    modality_legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                    markerfacecolor=color, markersize=8, 
                                    label=f'{modality_names[i]}'))

axes[0, -1].axis('off')
axes[0, -1].legend(handles=modality_legend_elements, title='Modality', 
                    loc='center left', fontsize=10)


In [29]:
#shuffling to aid plotting
idx = np.random.permutation(len(embedding_mod[modality]))
x_disp = x_disp[idx]
y_disp = y_disp[idx]
z_disp = z_disp[idx]
gripper = gripper[idx]


binary_map = ListedColormap(['red', 'green'])

x_mouvement = ((x_disp<0.15)|(x_disp>0.30))
x_stop = ((x_disp>0.15)|(x_disp<0.30))

x_disp_binary = x_disp[x_mouvement]
right_left_labels = np.where(x_disp_binary<0.15, 1,0)

y_disp_binary = y_disp[(y_disp > -0.5)]
front_back_labels = np.where((y_disp_binary > -0.25), 1,0)

for modality in modalities:
    embedding_mod[modality] = embedding_mod[modality][idx]

for row, action in enumerate(actions):
    actual_row = row + 1
    
    for col, modality in enumerate(modalities):
        ax = axes[actual_row, col]
        
        if action == 'right-left':
            scatter = ax.scatter(
                embedding_mod[modality][:, 0][x_mouvement],
                embedding_mod[modality][:, 1][x_mouvement],
                c=right_left_labels,
                cmap=binary_map,
                s=5,
                alpha=0.9
            )
            ax.set_title(f'{modality.upper()} - Right-Left')
            
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                pos = axes[actual_row, -1].get_position()
                cbar_height = max(0.02, pos.height * 0.6) 
                cbar_y = pos.y0 + (pos.height - cbar_height) / 2 
                cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
                cbar = plt.colorbar(scatter, cax=cbar_ax)
            
        elif action == 'front-back':
            scatter = ax.scatter(
                embedding_mod[modality][:, 0][(y_disp > -0.5)],
                embedding_mod[modality][:, 1][(y_disp > -0.5)],
                c=front_back_labels,
                cmap=binary_map,
                s=5,
                alpha=0.9
            )
            ax.set_title(f'{modality.upper()} -Front-Back')
            
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                pos = axes[actual_row, -1].get_position()
                cbar_height = max(0.02, pos.height * 0.6) 
                cbar_y = pos.y0 + (pos.height - cbar_height) / 2
                cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
                cbar = plt.colorbar(scatter, cax=cbar_ax)
            
        elif action == 'up-down':
            scatter = ax.scatter(
                embedding_mod[modality][:, 0][(gripper>0.2)&(x_disp<0)],
                embedding_mod[modality][:, 1][(gripper>0.2)&(x_disp<0)],
                c=gripper[(gripper>0.2)&(x_disp<0)],
                cmap='plasma',
                s=5,
                alpha=0.9
            )
            ax.set_title(f'{modality.upper()} - Down & Grip')
            
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                pos = axes[actual_row, -1].get_position()
                cbar_height = max(0.02, pos.height * 0.6) 
                cbar_y = pos.y0 + (pos.height - cbar_height) / 2
                cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
                cbar = plt.colorbar(scatter, cax=cbar_ax)
            
        elif action == 'gripper':
            scatter = ax.scatter(
                embedding_mod[modality][:, 0],
                embedding_mod[modality][:, 1],
                c=gripper,
                cmap='plasma',
                s=5,
                alpha=0.7
            )
            ax.set_title(f'{modality.upper()} - Gripper')
            
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                pos = axes[actual_row, -1].get_position()
                cbar_height = max(0.02, pos.height * 0.6) 
                cbar_y = pos.y0 + (pos.height - cbar_height) / 2
                cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
                cbar = plt.colorbar(scatter, cax=cbar_ax)
            
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)

plt.suptitle('UMAP Visualization - Modalities and Attributes', fontsize=20, fontweight='bold', y=0.98)
plt.savefig("graphs/with_act/Umap_shuffle_cosine.png")
plt.close()